# calcfinc in ten minutes

Financial ratios you can defend: every number comes with its **formula**, the **exact facts** it used, the
**filing** each fact came from, and a **reason** when something is missing.

This notebook runs on built-in sample data (no account, no download). Section 10 is an optional live SEC
download, section 11 shows an Indian exchange filing on sample data, and section 12 runs two real TCS filings
that are included in the repository. Needs Python 3.11 or later (Colab has it).

In [ ]:
%pip install -q calcfinc

## 1. Load some figures

Three fiscal years for a made-up company, each figure tagged with the filing it came from and the date it was
reported. Numbers are strings (or ints or `Decimal`s) on purpose: floats are refused, because `0.1` as a float
is not 0.1.

In [ ]:
from calcfinc import FinancialEngine

figures = {"FY2024": (900, 80, 450, 1800), "FY2025": (1000, 100, 500, 2000), "FY2026": (1200, 150, 600, 2400)}
metrics = ("revenue", "net_profit", "total_equity", "total_assets")
rows = [{"entity": "Acme", "metric": metric, "period": year, "value": str(value), "currency": "USD",
         "source": f"Acme 10-K {year}", "reported_at": f"{year[2:]}-11-15"}
        for year, values in figures.items() for metric, value in zip(metrics, values, strict=True)]

eng = FinancialEngine.from_records(rows)
eng.periods("Acme")

## 2. Ask a question, get an answer you can check

`get_ratio` returns a result object, not a bare number: the value, the formula, and the facts behind it.

In [ ]:
roe = eng.get_ratio("Acme", "roe")
roe.value        # -> 25
roe.formula      # -> 100 * net_profit / total_equity
roe.period       # -> FY2026

In [ ]:
roe.to_frame()     # the audit trail: one row per input fact, values still exact Decimals (needs pandas)

## 3. When something is missing, you are told

This company has no cash flow statement. A typical library would return `NaN` or `0`. calcfinc returns `None`
and says exactly what is missing.

In [ ]:
fcf = eng.get_ratio("Acme", "free_cash_flow")
fcf.value                 # -> None
print(fcf.limitations[0])

## 4. Growth and CAGR keep their lineage

Each input is the fact from its own year, with its own filing and date.

In [ ]:
growth = eng.get_growth("Acme", "revenue")
growth.value                               # -> 20
growth.to_frame()

In [ ]:
cagr = eng.get_cagr("Acme", "revenue")     # FY2024 -> FY2026, two years
round(cagr.value, 4)                       # -> 15.4701

In [ ]:
# a margin's growth traces back to the underlying revenue and profit of each year, not to "the margin"
eng.get_growth("Acme", "net_profit_margin").to_frame()

## 5. Restatements are kept, and mixing vintages is flagged

If a company restates a figure, both versions stay in the store and the latest wins. When one result mixes
figures first reported in filings far apart, it tells you.

In [ ]:
restated = rows + [{"entity": "Acme", "metric": "revenue", "period": "FY2026", "value": "1150", "currency": "USD",
                    "source": "Acme 10-K/A FY2026", "reported_at": "2027-09-01"}]
eng2 = FinancialEngine.from_records(restated)
eng2.get_metric("Acme", "revenue").value             # -> 1150
[(f.value, f.reported_at.isoformat()) for f in eng2.repos.facts.list_facts(1, metric="revenue") if f.financial_year == 2026]

In [ ]:
margin = eng2.get_ratio("Acme", "net_profit_margin")
margin.limitations

## 6. Exact arithmetic

Python floats cannot represent 0.1. calcfinc uses `Decimal` throughout.

In [ ]:
0.1 + 0.2                                    # the float answer

In [ ]:
eng.calculate("0.1 + 0.2").value             # -> 0.3

In [ ]:
from calcfinc.loaders import LoadError

try:
    FinancialEngine.from_records([{"entity": "A", "metric": "revenue", "period": "FY2026", "value": 0.1, "currency": "USD"}])
except LoadError as error:
    print(error.errors[0])

## 7. A bank is not a factory

Interest is a bank's cost of doing business, so "interest cover" and "debt to equity" mislead for a bank.
Declare the sector (or let calcfinc infer it from bank-specific lines) and those ratios return `None` with the
reason, while ROE, which does apply, still works.

In [ ]:
facts = {"revenue": "650", "pbt_before_exceptional": "100", "finance_costs": "250", "net_profit": "80",
         "total_equity": "400", "total_assets": "8000", "borrowings_noncurrent": "200"}
firm_rows = [{"entity": "Firm", "metric": m, "period": "FY2026", "value": v, "currency": "USD"} for m, v in facts.items()]

as_company = FinancialEngine.from_records(firm_rows)
as_bank = FinancialEngine.from_records(firm_rows, sector="bank")
as_company.get_ratio("Firm", "interest_coverage").value     # -> 1.4

In [ ]:
held_back = as_bank.get_ratio("Firm", "interest_coverage")
held_back.value                                              # -> None
print(held_back.limitations[0])
as_bank.get_ratio("Firm", "roe").value                       # -> 20

## 8. Define your own ratio

Ratios are formula strings in an open registry. DSCR has no single universal definition, so calcfinc does not
build one in. Define yours; the formula is what the result shows.

In [ ]:
from calcfinc import RatioSpec, StatementType, register_metric, register_ratio

register_metric("principal_repaid", "currency", StatementType.CASH_FLOW, label="Loan principal repaid")
register_ratio(RatioSpec(
    "dscr", "x", "(net_profit + depreciation + finance_costs) / (principal_repaid + finance_costs)",
    label="Debt service coverage"))

loan = {"net_profit": "80", "depreciation": "20", "finance_costs": "10", "principal_repaid": "40"}
bakery = FinancialEngine.from_records(
    [{"entity": "Bakery", "metric": m, "period": "FY2026", "value": v, "currency": "GBP"} for m, v in loan.items()])
dscr = bakery.get_ratio("Bakery", "dscr")
dscr.value        # -> 2.2
dscr.formula

## 9. Check your data

`check` tests accounting identities (assets = liabilities + equity, and so on). A failure is reported, never
"fixed".

In [ ]:
odd = FinancialEngine.from_records([{"entity": "X", "metric": m, "period": "FY2026", "value": v, "currency": "USD"}
                                    for m, v in (("total_assets", "100"), ("total_liabilities", "60"), ("total_equity", "30"))])
result = odd.check("X")[0]
result.failed         # -> ['assets_eq_liabilities_plus_equity']

## 10. A real filing from the SEC (optional, needs internet)

The SEC asks for a real contact in the request. Put yours in below. This downloads Apple's data once (about 4 MB)
and asks for return on equity from its 10-K.

In [ ]:
from calcfinc import SqliteRepositories
from calcfinc.adapters import sec_companyfacts as sec

CONTACT = "Your Name you@example.com"      # <- replace with your own name and email

raw = sec.fetch_companyfacts(320193, CONTACT)         # Apple's CIK
repos = SqliteRepositories(":memory:")
sec.load_companyfacts(repos, sec.read_companyfacts(raw), ticker="AAPL")
apple = FinancialEngine(repos)

roe = apple.get_ratio("AAPL", "roe", period="FY2025")
print(roe.value, roe.unit, roe.formula)
roe.to_frame()

In [ ]:
# follow a fact back to the filing it came from
source = repos.sources.get(roe.inputs[0].source_id)
print(source.document_title, source.uri)

# something Apple does not report: an answer with a reason, not a guess
print(apple.get_ratio("AAPL", "interest_coverage", period="FY2025").limitations)

## 11. An Indian exchange filing (Ind-AS XBRL)

The Indian adapter reads the exchange's `.xbrl` result files: April-March fiscal year, rupees, consolidated and
standalone kept apart, and the `india.*` ratios in Schedule III and ICAI forms.

The exchange's files cannot be redistributed, so the next cell writes five small **made-up** quarterly filings
for a company called SYNTH (real files carry many more lines but have the same layout). Figures are in crore
rupees (1 crore = 10 million).

In [ ]:
import tempfile
from pathlib import Path

CRORE = 10**7
folder = Path(tempfile.mkdtemp())


def write_filing(name, quarter, year_start, revenue, profit, assets, liabilities, equity, annual=None):
    """One exchange-style filing: the quarter (context OneD), the year to date (FourD, only in the March
    filing) and the balance sheet at the quarter end (OneI)."""
    start, end = quarter
    entity = '<xbrli:entity><xbrli:identifier scheme="syn">SYNTH</xbrli:identifier></xbrli:entity>'

    def period(a, b):
        return f"<xbrli:period><xbrli:startDate>{a}</xbrli:startDate><xbrli:endDate>{b}</xbrli:endDate></xbrli:period>"

    def fact(tag, context, crore):
        return f'<in-capmkt:{tag} contextRef="{context}" unitRef="INR">{crore * CRORE}</in-capmkt:{tag}>'

    lines = [
        '<?xml version="1.0" encoding="UTF-8"?>',
        '<xbrli:xbrl xmlns:xbrli="http://www.xbrl.org/2003/instance" xmlns:in-capmkt="http://example.test/in-capmkt">',
        f'<xbrli:context id="OneD">{entity}{period(start, end)}</xbrli:context>',
        f'<xbrli:context id="OneI">{entity}<xbrli:period><xbrli:instant>{end}</xbrli:instant></xbrli:period></xbrli:context>',
        '<xbrli:unit id="INR"><xbrli:measure>iso4217:INR</xbrli:measure></xbrli:unit>',
        fact("RevenueFromOperations", "OneD", revenue), fact("ProfitLossForPeriod", "OneD", profit),
        fact("Assets", "OneI", assets), fact("Liabilities", "OneI", liabilities), fact("Equity", "OneI", equity)]
    if annual:
        lines.insert(3, f'<xbrli:context id="FourD">{entity}{period(year_start, end)}</xbrli:context>')
        lines += [fact("RevenueFromOperations", "FourD", annual[0]), fact("ProfitLossForPeriod", "FourD", annual[1])]
    (folder / name).write_text("\n".join(lines + ["</xbrli:xbrl>"]), encoding="utf-8")


#             file                          quarter                      year starts  revenue profit assets liabilities equity
write_filing("31-Mar-2025_Consolidated.xbrl", ("2025-01-01", "2025-03-31"), "2024-04-01", 260, 26, 2000, 1500, 500, annual=(1000, 100))
write_filing("30-Jun-2025_Consolidated.xbrl", ("2025-04-01", "2025-06-30"), "2025-04-01", 250, 27, 2100, 1580, 520)
write_filing("30-Sep-2025_Consolidated.xbrl", ("2025-07-01", "2025-09-30"), "2025-04-01", 260, 28, 2160, 1612, 548)
write_filing("31-Dec-2025_Consolidated.xbrl", ("2025-10-01", "2025-12-31"), "2025-04-01", 270, 29, 2260, 1683, 577)
write_filing("31-Mar-2026_Consolidated.xbrl", ("2026-01-01", "2026-03-31"), "2025-04-01", 320, 36, 2400, 1800, 600, annual=(1100, 120))
sorted(p.name for p in folder.glob("*.xbrl"))

In [ ]:
from calcfinc import SqliteRepositories
from calcfinc.adapters import ind_as_xbrl

repos = SqliteRepositories(":memory:")
ind_as_xbrl.load_xbrl_files(repos, sorted(folder.glob("*.xbrl")), entity="SYNTH")   # the basis comes from the file name
india = FinancialEngine(repos)
india.periods("SYNTH")       # April-March years: the quarter Apr-Jun 2025 is FY2026 Q1

In [ ]:
india.get_metric("SYNTH", "revenue", period="FY2026").value / CRORE        # -> 1100

In [ ]:
# do the four quarters add up to the year? (a restated year beside unrestated quarters would show up here)
[check.failed for check in india.check_periods("SYNTH") if check.period == "FY2026"]       # -> [[]]

In [ ]:
india.get_ratio("SYNTH", "roe", period="FY2026").value                      # -> 20

In [ ]:
# the Schedule III form: profit after tax over AVERAGE equity (March 2025 and March 2026), not year-end equity
roe_india = india.get_ratio("SYNTH", "india.roe", period="FY2026")
round(roe_india.value, 4)                                                    # -> 21.8182
roe_india.formula

In [ ]:
roe_india.limitations      # an input that was not reported and was treated as zero is always announced

In [ ]:
roe_india.to_frame()       # net profit and both years' equity, each from its own filing

## 12. A real Indian company: TCS

Two real filings are in the repository (`examples/data/tcs`): Tata Consultancy Services' consolidated results for
the years ended March 2024 and March 2025, exactly as published on the exchange. They are public regulatory
filings, included here only for this demonstration (see the folder's README for provenance and terms).

The next cell downloads them from GitHub (or reads them from a clone of the repository) and asks for FY2025
return on equity in the Schedule III form.

In [ ]:
import tempfile
import urllib.request
from pathlib import Path

BASE = "https://raw.githubusercontent.com/Am1n1602/calcfinc/main/examples/data/tcs/"
TCS_FILES = ["31-Mar-2024_Financial_Results_Original_Consolidated.xbrl",
             "31-MAR-2025_Integrated_Filing-_Financials_Original_Consolidated.xbrl"]

clone = Path("examples/data/tcs")                         # present when you run this from a clone of the repository
if all((clone / name).exists() for name in TCS_FILES):
    tcs_folder = clone
else:                                                     # Colab and anywhere else: fetch the two files
    tcs_folder = Path(tempfile.mkdtemp())
    for name in TCS_FILES:
        urllib.request.urlretrieve(BASE + name, tcs_folder / name)
sorted(p.name for p in tcs_folder.glob("*.xbrl"))

In [ ]:
tcs_repos = SqliteRepositories(":memory:")
ind_as_xbrl.load_xbrl_files(tcs_repos, [tcs_folder / name for name in TCS_FILES], entity="TCS")
tcs = FinancialEngine(tcs_repos)

tcs_roe = tcs.get_ratio("TCS", "india.roe", period="FY2025")
round(tcs_roe.value, 4)         # -> 52.1642

In [ ]:
tcs_roe.formula

In [ ]:
tcs_roe.to_frame()              # profit and both years' equity, each traced to its own filing

In [ ]:
# which filing did each figure come from, and what did the library have to assume?
for fact in tcs_roe.inputs:
    print(fact.metric, fact.period, fact.value, tcs_repos.sources.get(fact.source_id).document_title)
tcs_roe.limitations

In [ ]:
# the plain ROE divides by year-end equity only, so it differs from the average-equity form above
round(tcs.get_ratio("TCS", "roe", period="FY2025").value, 4)                # -> 50.9517

For other companies or periods, download the files from the exchange yourself and point `load_xbrl_files` at them.
Original and Revision filings are loaded in the right order, and a bank's placeholder zeros are not loaded as
real zeros. See the [adapters page](https://github.com/Am1n1602/calcfinc/blob/main/docs/adapters.md).

## Where next

- [Manual](https://github.com/Am1n1602/calcfinc/blob/main/docs/manual.md): every question you can ask, periods, custom ratios,
  running it in production.
- [Ratio reference](https://github.com/Am1n1602/calcfinc/blob/main/docs/ratios.md): all ratios and their formulas.
- [Adapters and real-data results](https://github.com/Am1n1602/calcfinc/blob/main/docs/adapters.md).

Results are calculations, not investment advice.